In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [17]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import copy

In [2]:
CLEANED_PATH = r"C:\glassIDS\data\processed\cicids2017_cleaned.csv"
df = pd.read_csv(CLEANED_PATH)

feature_cols = [c for c in df.columns if c not in ("Label", "is_attack", "source_file", "Destination Port")]
X = df[feature_cols]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [3]:
# --- Isolate ONLY benign rows from the training split ---
benign_mask_train = (y_train == "BENIGN")
X_train_benign = X_train[benign_mask_train]

# Carve a validation split out of the benign-only data
X_train_ae, X_val_ae = train_test_split(
    X_train_benign, test_size=0.15, random_state=42
)

print(f"Benign-only training rows: {len(X_train_ae)}")
print(f"Benign-only validation rows: {len(X_val_ae)}")

Benign-only training rows: 1424638
Benign-only validation rows: 251407


In [6]:
#Sanity check
print(y_train[benign_mask_train].unique())  # should print only ['BENIGN']

<ArrowStringArray>
['BENIGN']
Length: 1, dtype: str


In [4]:
# --- Fresh scaler, fit ONLY on benign training data ---
ae_scaler = StandardScaler()
X_train_ae_scaled = ae_scaler.fit_transform(X_train_ae)
X_val_ae_scaled = ae_scaler.transform(X_val_ae)

# Transform the full (mixed) test set too, for evaluation later — fit never touches this
X_test_scaled_ae = ae_scaler.transform(X_test)

In [12]:
class BenignFlowDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

BATCH_SIZE = 512
train_loader = DataLoader(BenignFlowDataset(X_train_ae_scaled), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(BenignFlowDataset(X_val_ae_scaled), batch_size=BATCH_SIZE, shuffle=False)

In [13]:
class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_size),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Autoencoder(input_size=X_train_ae_scaled.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

MAX_EPOCHS = 50
PATIENCE = 5
best_val_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        reconstructed = model(batch)
        loss = criterion(reconstructed, batch)   # compare output to its OWN input
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            val_loss += loss.item() * batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            break

model.load_state_dict(best_state)

Epoch 1: train_loss=0.213606, val_loss=0.055662
Epoch 2: train_loss=0.108251, val_loss=0.053670
Epoch 3: train_loss=0.080371, val_loss=0.049717
Epoch 4: train_loss=0.067713, val_loss=0.085930
Epoch 5: train_loss=0.055121, val_loss=0.022815
Epoch 6: train_loss=0.046349, val_loss=0.024121
Epoch 7: train_loss=0.054590, val_loss=0.025073
Epoch 8: train_loss=0.045274, val_loss=0.018392
Epoch 9: train_loss=0.042917, val_loss=0.011634
Epoch 10: train_loss=0.043094, val_loss=0.017796
Epoch 11: train_loss=0.034731, val_loss=0.020132
Epoch 12: train_loss=0.038003, val_loss=0.050292
Epoch 13: train_loss=0.033972, val_loss=0.011574
Epoch 14: train_loss=0.028933, val_loss=0.028916
Epoch 15: train_loss=0.040056, val_loss=0.011013
Epoch 16: train_loss=0.023801, val_loss=0.009659
Epoch 17: train_loss=0.032715, val_loss=0.011768
Epoch 18: train_loss=0.021801, val_loss=0.027367
Epoch 19: train_loss=0.023442, val_loss=0.021481
Epoch 20: train_loss=0.024535, val_loss=0.010418
Epoch 21: train_loss=0.017263

<All keys matched successfully>

In [19]:
model.eval()
with torch.no_grad():
    train_benign_tensor = torch.tensor(X_train_ae_scaled, dtype=torch.float32).to(device)
    reconstructed_train = model(train_benign_tensor)
    # Per-row MSE, not one number averaged across everything — this is what the threshold gets built from
    train_recon_error = ((reconstructed_train - train_benign_tensor) ** 2).mean(dim=1).cpu().numpy()

THRESHOLD_PERCENTILE = 95
threshold = np.percentile(train_recon_error, THRESHOLD_PERCENTILE)
print(f"Reconstruction error at {THRESHOLD_PERCENTILE}th percentile (benign training data): {threshold:.6f}")
print(f"Mean: {train_recon_error.mean():.6f}, Std: {train_recon_error.std():.6f}")

Reconstruction error at 95th percentile (benign training data): 0.020175
Mean: 0.015881, Std: 3.967990


In [20]:
top_errors = pd.Series(train_recon_error).nlargest(10)
print(top_errors)

# Pull the actual raw (unscaled) feature values for these rows
worst_idx = top_errors.index
print(X_train_ae.iloc[worst_idx][['Flow Duration', 'Flow Bytes/s', 'Total Fwd Packets']])

931968     2602.089600
1041037    2560.970703
644031     2557.993896
424894     1062.770264
945107      786.761841
319381      422.330627
639871      317.161163
517315      307.190002
1029140     223.729004
606695      220.895035
dtype: float32
         Flow Duration  Flow Bytes/s  Total Fwd Packets
1520603       79999626  7.200034e+00                  6
1785102            273  9.450549e+05                  2
1794685            227  8.810573e+05                  2
1520609       87434104  3.319185e+02                235
1826946         415806  1.616138e+03                 24
1545069      119990161  3.465092e+06             142064
1883227         418893  5.156448e+03                 24
1521575        3636158  4.546007e+03                246
1111817      111485719  3.588727e+06             115727
452761       110291452  2.066170e+06              77620


In [21]:
model.eval()
with torch.no_grad():
    test_tensor = torch.tensor(X_test_scaled_ae, dtype=torch.float32).to(device)
    reconstructed_test = model(test_tensor)
    test_recon_error = ((reconstructed_test - test_tensor) ** 2).mean(dim=1).cpu().numpy()

results_df = pd.DataFrame({
    'true_label': y_test.values,
    'recon_error': test_recon_error,
    'predicted_anomaly': test_recon_error > threshold
})

# False positive rate on benign test rows
benign_results = results_df[results_df['true_label'] == 'BENIGN']
fpr = benign_results['predicted_anomaly'].mean()
print(f"False positive rate on BENIGN test rows: {fpr:.4f}")

# Detection rate (recall) per attack class
attack_detection = (
    results_df[results_df['true_label'] != 'BENIGN']
    .groupby('true_label')['predicted_anomaly']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'detection_rate', 'count': 'n_flows'})
    .sort_values('detection_rate')
)
print(attack_detection)

False positive rate on BENIGN test rows: 0.0498
                            detection_rate  n_flows
true_label                                         
Web Attack - Sql Injection        0.000000        4
SSH-Patator                       0.000000      644
FTP-Patator                       0.000843     1186
PortScan                          0.012459    18139
Web Attack - XSS                  0.030769      130
Bot                               0.035897      390
Web Attack - Brute Force          0.044218      294
DDoS                              0.639066    25603
DoS slowloris                     0.648097     1077
DoS GoldenEye                     0.732620     2057
DoS Hulk                          0.902253    34569
DoS Slowhttptest                  0.966539     1046
Heartbleed                        1.000000        2
Infiltration                      1.000000        7


In [22]:
combined = pd.read_csv(r"C:\glassIDS\data\processed\self_captured\combined_labeled.csv")
print(combined.shape)
print(combined['category'].value_counts())

(66594, 83)
category
attack_portscan             65535
attack_slowdos               1000
attack_probe                   41
attack_bruteforce              14
benign_control                  2
benign_control_typed            1
benign_control_logstream        1
Name: count, dtype: int64


In [23]:
#Re-add the duplicate column this model was trained on (same fix the XGBoost scoring needed)
combined['Fwd Header Length.1'] = combined['Fwd Header Length']

X_combined = combined[feature_cols]

assert len(X_combined.columns) == len(feature_cols), "Column count mismatch"

X_combined_scaled = ae_scaler.transform(X_combined)

model.eval()
with torch.no_grad():
    combined_tensor = torch.tensor(X_combined_scaled, dtype=torch.float32).to(device)
    reconstructed_combined = model(combined_tensor)
    combined_recon_error = ((reconstructed_combined - combined_tensor) ** 2).mean(dim=1).cpu().numpy()

combined['recon_error'] = combined_recon_error
combined['predicted_anomaly'] = combined_recon_error > threshold

detection_by_category = (
    combined.groupby('category')['predicted_anomaly']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'detection_rate', 'count': 'n_flows'})
)
print(detection_by_category)

                          detection_rate  n_flows
category                                         
attack_bruteforce                    1.0       14
attack_portscan                      1.0    65535
attack_probe                         1.0       41
attack_slowdos                       1.0     1000
benign_control                       1.0        2
benign_control_logstream             1.0        1
benign_control_typed                 1.0        1


In [24]:
print(f"Threshold: {threshold}")
print(combined.groupby('category')['recon_error'].describe())
print(f"\nAny Inf in recon_error: {np.isinf(combined_recon_error).any()}")
print(f"Any NaN in recon_error: {np.isnan(combined_recon_error).any()}")

Threshold: 0.020175041630864143
                            count          mean          std           min  \
category                                                                     
attack_bruteforce            14.0  6.869122e+01    10.959179  5.528598e+01   
attack_portscan           65535.0  2.116967e+01     0.210519  2.108353e+01   
attack_probe                 41.0  1.153041e+01    16.084105  8.794373e+00   
attack_slowdos             1000.0  2.086906e+02    75.143066  1.423593e+02   
benign_control                2.0  9.806805e+03  3822.484619  7.103899e+03   
benign_control_logstream      1.0  2.715699e+09          NaN  2.715699e+09   
benign_control_typed          1.0  1.308792e+04          NaN  1.308792e+04   

                                   25%           50%           75%  \
category                                                             
attack_bruteforce         6.490565e+01  6.644623e+01  6.644721e+01   
attack_portscan           2.109583e+01  2.110783e+01  2

In [25]:
print(feature_cols == list(X_train_ae.columns))

True


In [26]:
model.eval()
with torch.no_grad():
    sample_scan_row = X_combined_scaled[combined['category'] == 'attack_portscan'][:1]
    sample_tensor = torch.tensor(sample_scan_row, dtype=torch.float32).to(device)
    reconstructed_sample = model(sample_tensor).cpu().numpy()[0]

squared_errors = (sample_scan_row[0] - reconstructed_sample) ** 2
error_breakdown = pd.DataFrame({
    'feature': feature_cols,
    'scaled_input_value': sample_scan_row[0],
    'squared_error': squared_errors
}).sort_values('squared_error', ascending=False)

print(error_breakdown.head(10))

                  feature  scaled_input_value  squared_error
49         ECE Flag Count           -0.018205     915.618517
44         RST Flag Count           54.988264     611.092297
43         SYN Flag Count            8.416893      65.302883
46         ACK Flag Count            1.565431       5.791984
37      Min Packet Length            1.264332       2.907818
23            Fwd IAT Min           -0.131468       1.615262
16           Flow IAT Std           -0.280915       1.529655
45         PSH Flag Count           -0.605193       1.239539
17           Flow IAT Max           -0.322412       1.217666
9   Bwd Packet Length Max           -0.444322       1.116930


In [27]:
with torch.no_grad():
    sample_log_row = X_combined_scaled[combined['category'] == 'benign_control_logstream'][:1]
    sample_tensor2 = torch.tensor(sample_log_row, dtype=torch.float32).to(device)
    reconstructed_sample2 = model(sample_tensor2).cpu().numpy()[0]

squared_errors2 = (sample_log_row[0] - reconstructed_sample2) ** 2
error_breakdown2 = pd.DataFrame({
    'feature': feature_cols,
    'scaled_input_value': sample_log_row[0],
    'squared_error': squared_errors2
}).sort_values('squared_error', ascending=False)

print(error_breakdown2.head(10))

                  feature  scaled_input_value  squared_error
57      Fwd Avg Bulk Rate       442786.272908   1.960560e+11
55     Fwd Avg Bytes/Bulk        57803.428571   3.442933e+09
34      Bwd Header Length            0.038639   1.936134e+09
68   min_seg_size_forward            0.002966   1.545219e+09
50          Down/Up Ratio           -0.266426   6.251167e+08
29          Fwd PSH Flags        24178.684680   5.890521e+08
37      Min Packet Length            1.711065   4.764593e+08
10  Bwd Packet Length Min            0.191644   4.664758e+08
33      Fwd Header Length            0.012130   4.494651e+08
54    Fwd Header Length.1            0.012130   4.180127e+08
